# PATSTAT Citation Trend — yearly incoming citations per application

The twin of `PatentView/notebook/patent_citation_trend.ipynb`: the same edges as `patstat_citation`,
resolved **by citing year** instead of by window, so the two files answer different questions from the
same data and their totals must agree (checked at the end).

Clock: `cite_year` = filing year of the citing application, `yrs_since_filing = cite_year - filing_year`
of the cited application (`ps.EDGE_WHERE`: age >= 0, not replenished).

## Output
`PATSTAT/output/patstat_citation_trend.parquet` — one row per (application, citing year) with >= 1 citation:
`appln_id, filing_year, cite_year, yrs_since_filing, C, C_examiner, C_applicant, C_other, uniqueC`
(`uniqueC` = distinct citing applications that year; a citing application has one filing year, so the yearly
`uniqueC` sum to the windowed `uniqueC_all` exactly).

In [ ]:
import os, sys, gc, time
import numpy as np, pandas as pd
import pyarrow as pa, pyarrow.parquet as pq
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/PATSTAT')
import ps_common as ps
OUT = ps.OUT
OUT_FP = ps.out('patstat_citation_trend.parquet')
ps.preflight('patstat_citation_trend')

REF, CIT = ps.out('patstat_reference.parquet'), ps.out('patstat_citation.parquet')
con = ps.connect()

## 1. Yearly series

In [ ]:
%%time
con.execute(f"""COPY (
  SELECT cited_id AS appln_id, cited_year AS filing_year, citing_year AS cite_year, age AS yrs_since_filing,
         count(*) AS C,
         {', '.join(f"count(*) FILTER (WHERE bucket = '{b}') AS C_{b}" for b in ps.BUCKETS)},
         count(DISTINCT citing_id) AS uniqueC
  FROM read_parquet('{REF}') WHERE {ps.EDGE_WHERE}
  GROUP BY 1, 2, 3, 4 ORDER BY 1, 3
) TO '{OUT_FP}' (FORMAT PARQUET, COMPRESSION ZSTD)""")
m = pq.ParquetFile(OUT_FP).metadata
print(f'WROTE {OUT_FP}  ({m.num_rows:,} rows, {os.path.getsize(OUT_FP)/1e6:.0f} MB)')

## 2. Reconcile with the windowed file

In [ ]:
%%time
# the yearly series must reproduce the windowed file exactly
chk = con.execute(f"""WITH t AS (SELECT appln_id, sum(C) AS C_all, sum(C) FILTER (WHERE yrs_since_filing <= 5) AS C_5,
                                sum(uniqueC) AS uniqueC_all, sum(uniqueC) FILTER (WHERE yrs_since_filing <= 5) AS uniqueC_5
                         FROM read_parquet('{OUT_FP}') GROUP BY 1)
  SELECT count(*) AS apps,
         count(*) FILTER (WHERE t.C_all <> c.C_all) AS C_all_mismatch, count(*) FILTER (WHERE coalesce(t.C_5, 0) <> c.C_5) AS C_5_mismatch,
         count(*) FILTER (WHERE t.uniqueC_all <> c.uniqueC_all) AS uniqueC_all_mismatch
  FROM t FULL JOIN read_parquet('{CIT}') c USING (appln_id)""").fetchdf()
display(chk); assert chk.C_all_mismatch[0] == 0 and chk.C_5_mismatch[0] == 0 and chk.uniqueC_all_mismatch[0] == 0
print('yearly series reproduces C_5 / C_all / uniqueC_all for every application: YES')
prof = con.execute(f"""SELECT yrs_since_filing, sum(C) AS C, sum(uniqueC) AS uniqueC, sum(C_examiner) AS examiner, sum(C_applicant) AS applicant
  FROM read_parquet('{OUT_FP}') WHERE yrs_since_filing <= 30 GROUP BY 1 ORDER BY 1""").fetchdf()
display(prof.head(12))
con.close()